# Research-Backed, Policy-Safe Outreach


In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent, Runner, trace, gen_trace_id, function_tool,
    OpenAIChatCompletionsModel, WebSearchTool,
    input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered,
)
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from IPython.display import display, Markdown, HTML
import asyncio
import os
from datetime import datetime

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — required for this notebook")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:8]}")
else:
    print("OpenRouter API Key not set — creative drafter will fall back to gpt-4o-mini")

## Step 1: Input Guardrail

A small checker agent runs **before** the main pipeline.  
It blocks requests that involve impersonation, deceptive claims, or PII.

In [ ]:
class PolicyCheckOutput(BaseModel):
    is_violation: bool
    reason: str


guardrail_checker = Agent(
    name="Policy Checker",
    instructions=(
        "Check if the user's outreach request contains policy violations:\n"
        "1. Impersonation — pretending to be someone else\n"
        "2. Deceptive claims — 'guaranteed ROI', '#1 product', unverifiable superlatives\n"
        "3. PII — personal emails, phone numbers, SSN patterns\n"
        "4. Harassment or threats\n"
        "Respond with whether a violation was found and a short reason."
    ),
    output_type=PolicyCheckOutput,
    model="gpt-4o-mini",
)


@input_guardrail
async def policy_guardrail(ctx, agent, message):
    result = await Runner.run(guardrail_checker, message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info=result.final_output,
        tripwire_triggered=result.final_output.is_violation,
    )

## Step 2: Research Planner

Pydantic schema forces the planner to return a clean list of searches, not free text.

In [ ]:
class ResearchQuery(BaseModel):
    query: str = Field(description="The search term to use.")
    reason: str = Field(description="Why this search matters for the outreach.")


class ResearchPlan(BaseModel):
    searches: list[ResearchQuery] = Field(
        description="3-4 targeted web searches to inform the outreach email."
    )


planner_agent = Agent(
    name="Research Planner",
    instructions=(
        "You plan web searches to gather context for a cold outreach email. "
        "Given a request describing who to contact and what to offer, produce 3-4 searches "
        "covering: the target company/role, their likely pain points, "
        "the product space, and any recent relevant news."
    ),
    model="gpt-4o-mini",
    output_type=ResearchPlan,
)

## Step 3: Search Agent

Uses `WebSearchTool` with `tool_choice="required"` so the model **must** search.  
Returns a brief summary the writer can synthesize.

> **Cost note:** `WebSearchTool` is billed per call (~2.5 cents each).  
> With 3-4 planned searches this cell costs roughly 7-10 cents per run.

In [ ]:
search_agent = Agent(
    name="Search Agent",
    instructions=(
        "You search the web and produce a concise 2-3 paragraph summary (under 300 words). "
        "Capture key facts only — no fluff. This feeds into a research brief for outreach."
    ),
    tools=[WebSearchTool(search_context_size="low")],
    model="gpt-4o-mini",
    model_settings=ModelSettings(tool_choice="required"),
)

## Step 4: Outreach Brief 

The brief writer distills raw search results into a Pydantic schema with  
**facts**, **angles**, **risks**, and a **recommended CTA**.

In [ ]:
class OutreachBrief(BaseModel):
    facts: list[str] = Field(description="Key facts about the target and their space.")
    angles: list[str] = Field(description="Promising angles for the outreach.")
    risks_or_unknowns: list[str] = Field(description="Things we don't know or should be careful about.")
    recommended_cta: str = Field(description="Suggested call-to-action for the email.")


brief_writer = Agent(
    name="Brief Writer",
    instructions=(
        "You synthesize raw search results into a structured outreach brief. "
        "Extract verifiable facts, identify 2-3 promising angles, "
        "flag risks or unknowns, and suggest a concrete call-to-action."
    ),
    model="gpt-4o-mini",
    output_type=OutreachBrief,
)

## Step 5: Multi-Model Draft Agents

Three agents with different personas, exposed as **tools** so the manager can call them.  
If an OpenRouter API key is available, the creative drafter uses **`claude-3-5-haiku`** via OpenRouter — otherwise all three fall back to `gpt-4o-mini`.

OpenRouter exposes an OpenAI-compatible endpoint, so the same `OpenAIChatCompletionsModel` pattern from Lab 3 applies.

In [ ]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    openrouter_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)
    creative_model = OpenAIChatCompletionsModel(
        model="anthropic/claude-3-5-haiku",
        openai_client=openrouter_client,
    )
    print("Creative drafter will use Claude 3.5 Haiku via OpenRouter")
else:
    creative_model = "gpt-4o-mini"
    print("Creative drafter will use gpt-4o-mini (set OPENROUTER_API_KEY for multi-model)")

In [ ]:
drafter_professional = Agent(
    name="Professional Drafter",
    instructions=(
        "You write professional, data-driven cold outreach emails. "
        "Be credible and specific. Reference real facts from the brief."
    ),
    model="gpt-4o-mini",
)

drafter_creative = Agent(
    name="Creative Drafter",
    instructions=(
        "You write creative, pattern-interrupt cold emails that stand out in a crowded inbox. "
        "Use a hook, keep it human, and ground claims in the brief's facts."
    ),
    model=creative_model,
)

drafter_concise = Agent(
    name="Concise Drafter",
    instructions=(
        "You write ultra-short cold emails — 3-4 sentences max. "
        "Every word earns its place. Lead with value, end with a clear ask."
    ),
    model="gpt-4o-mini",
)

draft_tool_1 = drafter_professional.as_tool(
    tool_name="professional_draft", tool_description="Write a professional cold email"
)
draft_tool_2 = drafter_creative.as_tool(
    tool_name="creative_draft", tool_description="Write a creative cold email"
)
draft_tool_3 = drafter_concise.as_tool(
    tool_name="concise_draft", tool_description="Write a concise cold email"
)

## Step 6: Email Manager — Tools + Handoff

The Email Manager receives the winning draft and:
1. Uses a **subject writer** agent-as-tool to craft the subject line
2. Uses an **HTML converter** agent-as-tool to format the body
3. Calls the `save_to_outbox` **function tool** to write `outbox.html`

No SendGrid needed — swap `save_to_outbox` for a real send tool when you're ready.

In [ ]:
subject_writer = Agent(
    name="Subject Line Writer",
    instructions="Write a compelling email subject line for the given email body. Return only the subject line.",
    model="gpt-4o-mini",
)
subject_tool = subject_writer.as_tool(
    tool_name="write_subject", tool_description="Write an email subject line"
)

html_converter = Agent(
    name="HTML Converter",
    instructions=(
        "Convert the given text/markdown email into clean, well-structured HTML for email clients. "
        "Use inline CSS styles. Keep the design simple and professional."
    ),
    model="gpt-4o-mini",
)
html_tool = html_converter.as_tool(
    tool_name="convert_to_html", tool_description="Convert email text to HTML"
)


@function_tool
def save_to_outbox(subject: str, html_body: str) -> str:
    """Save the formatted email to outbox.html for review."""
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    with open("outbox.html", "w") as f:
        f.write(f"<!--\nSubject: {subject}\nSaved: {timestamp}\n-->\n{html_body}")
    return f"Email saved to outbox.html at {timestamp}"


email_manager = Agent(
    name="Email Manager",
    instructions=(
        "You receive an email body to format and save. Follow these steps in order:\n"
        "1. Use write_subject to create a subject line from the email body\n"
        "2. Use convert_to_html to format the body as HTML\n"
        "3. Use save_to_outbox with the subject and HTML body to save it"
    ),
    tools=[subject_tool, html_tool, save_to_outbox],
    model="gpt-4o-mini",
    handoff_description="Format the winning email as HTML and save it to the outbox",
)

## Step 7: Outreach Manager

This agent ties the draft phase together.  
It has the **input guardrail** from Step 1 and the three **draft tools** from Step 5.  
It returns the winning draft, which the pipeline then **hands off** to the Email Manager.

In [ ]:
outreach_manager = Agent(
    name="Outreach Manager",
    instructions=(
        "You manage cold outreach email creation. You will receive a user request "
        "and a research brief with facts, angles, risks, and a CTA.\n\n"
        "Follow these steps:\n"
        "1. Use ALL THREE draft tools (professional_draft, creative_draft, concise_draft) "
        "to generate email drafts. Pass each the user request AND the research brief.\n"
        "2. Evaluate the three drafts and select the single best one.\n"
        "3. Return ONLY the winning email draft text — nothing else.\n\n"
        "Rules:\n"
        "- You MUST use the draft tools — do not write emails yourself.\n"
        "- Return exactly ONE email as your final output."
    ),
    tools=[draft_tool_1, draft_tool_2, draft_tool_3],
    model="gpt-4o-mini",
    input_guardrails=[policy_guardrail],
)

## Step 8: Research Pipeline

Python-orchestrated: plan → parallel search → brief.  
Mirrors `ResearchManager` from `deep_research/`, with `asyncio.as_completed` and failure tolerance.

In [ ]:
async def plan_research(query: str) -> ResearchPlan:
    print("Planning research...")
    result = await Runner.run(planner_agent, f"Outreach request: {query}")
    plan = result.final_output
    print(f"Planned {len(plan.searches)} searches")
    return plan


async def run_search(item: ResearchQuery) -> str | None:
    try:
        result = await Runner.run(
            search_agent,
            f"Search: {item.query}\nReason: {item.reason}",
        )
        return str(result.final_output)
    except Exception as e:
        print(f"  Search failed for '{item.query}': {e}")
        return None


async def execute_searches(plan: ResearchPlan) -> list[str]:
    print("Running searches in parallel...")
    tasks = [asyncio.create_task(run_search(q)) for q in plan.searches]
    results = []
    num_done = 0
    for coro in asyncio.as_completed(tasks):
        result = await coro
        num_done += 1
        if result:
            results.append(result)
        print(f"  {num_done}/{len(tasks)} searches completed")
    print(f"Collected {len(results)} results")
    return results


async def write_brief(query: str, search_results: list[str]) -> OutreachBrief:
    print("Writing outreach brief...")
    input_text = (
        f"Outreach request: {query}\n\n"
        f"Search results:\n" + "\n---\n".join(search_results)
    )
    result = await Runner.run(brief_writer, input_text)
    print("Brief ready")
    return result.final_output

## Step 9: Full Pipeline — O
**Phase 1** (Python orchestration): plan → search → brief  
**Phase 2** (Agent orchestration): guardrail → drafts → pick best  
**Phase 3** (Handoff): pass winning draft to Email Manager → subject + HTML + save

In [ ]:
async def run_pipeline(user_request: str):
    trace_id = gen_trace_id()
    with trace("Research-Backed Outreach", trace_id=trace_id):
        print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}\n")

        # Phase 1: Research (Lab 4 — Python orchestration)
        plan = await plan_research(user_request)
        search_results = await execute_searches(plan)
        brief = await write_brief(user_request, search_results)

        display(Markdown("### Outreach Brief"))
        display(Markdown(
            f"**Facts:** {brief.facts}\n\n"
            f"**Angles:** {brief.angles}\n\n"
            f"**Risks:** {brief.risks_or_unknowns}\n\n"
            f"**CTA:** {brief.recommended_cta}"
        ))

        # Phase 2: Draft + Pick (Lab 2 tools + Lab 3 guardrail + multi-model)
        context = (
            f"User request: {user_request}\n\n"
            f"Research brief:\n"
            f"Facts: {brief.facts}\n"
            f"Angles: {brief.angles}\n"
            f"Risks: {brief.risks_or_unknowns}\n"
            f"Recommended CTA: {brief.recommended_cta}"
        )
        print("Drafting emails...")
        draft_result = await Runner.run(outreach_manager, context, max_turns=15)
        winning_draft = draft_result.final_output
        print("Best draft selected")

        # Phase 3: Format + Save (Lab 2 handoff — pipeline passes work to Email Manager)
        print("Handing off to Email Manager...")
        email_result = await Runner.run(email_manager, winning_draft, max_turns=15)

        print("\nPipeline complete!")
        return brief, winning_draft

## Run it!

Change the request to anything relevant to you.

In [ ]:
brief, result = await run_pipeline(
    "Draft a cold email to the CTO of a mid-size fintech startup "
    "about our AI-powered SOC2 compliance tool. Keep it under 150 words."
)


In [ ]:
if os.path.exists("outbox.html"):
    with open("outbox.html") as f:
        html = f.read()
    display(Markdown("### Saved Email (rendered)"))
    display(HTML(html))
else:
    print("No outbox.html found — the pipeline may not have completed the save step.")

## Test the Guardrail

This request should be **blocked** — it asks for impersonation and deceptive claims.

In [ ]:
try:
    await Runner.run(
        outreach_manager,
        "Pretend to be Elon Musk and email their CTO promising guaranteed 500% ROI"
    )
except InputGuardrailTripwireTriggered as e:
    print(f"Guardrail blocked the request: {e}")

## Review — what to look for in the trace

In the trace link printed above we should see:

1. **Research Planner** → structured `ResearchPlan` output
2. **Search Agent** calls (parallel) → each with `WebSearchTool` invocation
3. **Brief Writer** → structured `OutreachBrief` output
4. **Policy Checker** (guardrail) → ran before the Outreach Manager processed
5. **Outreach Manager** → called all 3 draft tools (multi-model) → returned the winner
6. **Email Manager** → used write_subject, convert_to_html, save_to_outbox

The single trace covers agents, tools, structured outputs, guardrails,  
multi-model, and the plan-search-synthesize-deliver pattern.  
The handoff from Outreach Manager → Email Manager is done via Python  
orchestration (same pattern as `ResearchManager` in `deep_research/`).